# Telecom Customer Data 
## Data Quality Assessment and Cleaning


### 1 · Setup & load

We keep an immutable copy of the raw data (df_raw) so the final before/after
comparison is grounded in the true original state. All cleaning happens on the
working copy `df`.

In [2]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

RAW_PATH = "test_datafile.csv"
df_raw = pd.read_csv(RAW_PATH)   # immutable reference
df = df_raw.copy()               # working copy we clean in place

print(f"Loaded {df_raw.shape[0]:,} rows  x  {df_raw.shape[1]} columns")
df_raw.head()

Loaded 5,050 rows  x  17 columns


,customer_id,age,gender,tenure_months,contract_type,monthly_charges,total_charges,internet_service,phone_service,avg_monthly_gb_used,num_support_tickets,avg_monthly_minutes,satisfaction_score,payment_method,num_additional_services,last_interaction_date,churned
0,TC-004711,32.87,Male,10.15,Month-to-month,69.24,656.42,DSL,Yes,11.70,4.00,324.00,7.80,bank transfer,2,2024-06-14,1
1,TC-000692,59.39,Female,3.45,Month-to-month,98.48,251.15,DSL,no,9.46,1.00,306.80,6.00,Electronic check,5,2024-06-23,1
2,TC-000066,62.34,male,1.39,Two year,94.35,120.78,Fiber optic,Yes,9.56,4.00,349.50,5.50,Bank transfer,0,2024-06-21,0
3,TC-003427,45.79,Female,67.61,Month-to-month,85.87,"5,834.73",Fiber optic,yes,3.15,1.00,258.20,4.70,Credit card,4,2024-06-21,1
4,TC-004821,39.63,F,27.32,One year,62.14,"1,626.23",DSL,Yes,28.80,0.00,335.80,12.30,Credit card,2,2024-06-19,0


In [9]:
# datatypes of each columns
df_raw.dtypes.to_frame("data_types")

,data_types
customer_id,object
age,float64
gender,object
tenure_months,float64
contract_type,object
monthly_charges,float64
total_charges,float64
internet_service,object
phone_service,object
avg_monthly_gb_used,float64


### 2 · Audit infrastructure

A small helpers power the whole notebook:

* `col_profile(frame)` — a per-column snapshot (dtype, # missing, # unique,
  numeric min/max). We snapshot the **raw** frame now; we snapshot the cleaned
  frame at the end. The two snapshots become the before/after table.


In [10]:
def col_profile(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for c in frame.columns:
        s = frame[c]
        rec = {
            "column": c,
            "dtype": str(s.dtype),
            "n_missing": int(s.isna().sum()),
            "n_unique": int(s.nunique(dropna=True)),
        }
        if pd.api.types.is_numeric_dtype(s) and s.notna().any():
            rec["min"] = round(float(s.min()), 2)
            rec["max"] = round(float(s.max()), 2)
        else:
            rec["min"] = rec["max"] = None
        rows.append(rec)
    return pd.DataFrame(rows).set_index("column")

profile_before = col_profile(df_raw)
profile_before

,dtype,n_missing,n_unique,min,max
column,,,,,
customer_id,object,0,5000,NaN,NaN
age,float64,10,4706,-1.00,999.00
gender,object,50,8,NaN,NaN
tenure_months,float64,11,4757,-12.00,500.00
contract_type,object,0,3,NaN,NaN
monthly_charges,float64,12,3740,-50.00,"9,999.00"
total_charges,float64,23,4935,0.66,"218,681.58"
internet_service,object,505,5,NaN,NaN
phone_service,object,0,6,NaN,NaN


The raw profile already reveals the trouble spots at a glance:

* **Missing values** in 10 columns.
* `gender`, `internet_service`, `phone_service`, `payment_method` carry far
  more unique values than they semantically should (case/abbreviation noise).
* Numeric **min/max are physically impossible**: `age` min `-1` / max `999`,
  `tenure_months` `-12` / `500`, `monthly_charges` `-50` / `9999`,
  `satisfaction_score` `-1.4` / `99`, `total_charges` up to `218,681`.

### 3 · Exact duplicate rows

**Detection.** Count byte-identical rows across *all* columns.

**Decision.** Drop them, keeping the first occurrence. Fully identical records
(same `customer_id` and every field) carry no new information and would
double-weight those customers in any downstream model or aggregate.

In [36]:
n_dupes = int(df.duplicated().sum())
dup_ids = df.loc[df.duplicated(keep=False), "customer_id"].nunique()
print(f"Exact duplicate rows: {n_dupes}  (covering {dup_ids} customer_ids)")

df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Rows after de-duplication: {len(df):,}")

Exact duplicate rows: 0  (covering 0 customer_ids)
Rows after de-duplication: 5,000


## 4 · Inconsistent categorical encodings

**Detection.** Inspect the value counts of every categorical column. The same
real-world category appears under several surface forms (different casing,
abbreviations).

**Decision.** Map each surface form to a single canonical label. We *do not*
touch genuine `NaN`s here (missingness is handled in §7) and we preserve
genuinely distinct categories:

| Column | Canonical values | Notes |
|---|---|---|
| `gender` | Male / Female / Other | `M`,`MALE`,`male` → Male, etc. `Other` is a real category, kept. |
| `internet_service` | Fiber optic / DSL / No | `fiber`→Fiber optic, `dsl`→DSL. **`No` = customer has no internet — a valid value, *not* missing.** |
| `phone_service` | Yes / No | `Y`/`yes`→Yes, `N`/`no`→No. |
| `payment_method` | Credit card / Bank transfer / Electronic check / Mailed check | `CC`→Credit card, `BT`→Bank transfer. |

Casing/abbreviation noise inflates cardinality and silently fragments groups
(`Male` and `male` would be treated as two segments), so canonicalisation is
mandatory before any grouping or encoding.